# 05 - MusicBrainz Artist Enrichment

## Goal

Enrich Ticketmaster artists with MusicBrainz metadata to support artist matching and concert recommendation.

## Tasks

- Extract unique artists from PostgreSQL
- Query MusicBrainz for artist matches
- Store canonical artist identifiers
- Review ambiguous or missing matches
- Prepare enriched artist data for the ranking engine

In [3]:
import os
from pathlib import Path
import pandas as pd
import requests

from dotenv import load_dotenv
from sqlalchemy import URL, text, create_engine

In [6]:
project_path = Path("..")

load_dotenv(project_path / ".env", override= True)
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
db_name = os.getenv("POSTGRES_DB")
db_port = os.getenv("POSTGRES_PORT")

assert db_user is not None
assert db_password is not None
assert db_name is not None
assert db_port is not None

In [7]:
database_url = URL.create(
    drivername= "postgresql+psycopg2",
    username= db_user,
    password= db_password,
    host= "localhost",
    port= int(db_port),
    database= db_name
)

engine= create_engine(database_url)

In [10]:
artists_df = pd.read_sql(
    """
    SELECT
        DISTINCT artist_name
    FROM events
    WHERE artist_name IS NOT NULL
    ORDER BY artist_name
    """,
    engine
)

artists_df.shape

(472, 1)

In [12]:
musicbrainz_url = "https://musicbrainz.org/ws/2/artist/"

headers = {
    "User-Agent": "GigRouteEurope/1.0"
}

In [19]:
test_artist = (artists_df).iloc[0]["artist_name"]

params = {
    "query": f'artist:"{test_artist}"',
    "fmt": "json",
    "limit": 5
}
response = requests.get(
    musicbrainz_url,
    params=params,
    headers=headers,
    timeout=30
)

response.status_code

200

In [27]:
artist_data = response.json()
artist_data.keys()

artist_data.get('artists', [])[:2]

[{'id': '6f4926ae-865e-483b-86f9-9ae3b9584507',
  'type': 'Person',
  'type-id': 'b6e035f4-3ce9-331c-97df-83397230b0df',
  'score': 100,
  'name': '54 Ultra',
  'sort-name': '54 Ultra',
  'life-span': {'ended': None}}]